# ECCI: From An EBSD Orientation To A Two-Beam Tilt

Electron channelling contrast imaging (ECCI) images dislocations without a TEM foil: a bulk
specimen in an SEM is tilted so that one strong reflection sits close to the Bragg condition while
its neighbours are suppressed — a **two-beam condition** — and the backscattered-electron signal
then carries the channelling contrast a dislocation core disturbs.

The natural starting point for that tilt is an orientation an EBSD system has already measured at
the same point: index the grain, choose the crystallographic direction you want on the beam, and
solve for the stage move that gets it there. That is exactly what `pytex.app.services.ecci`
implements, registered as two operations:

- `ecci.solve_workflow` — from an EBSD-measured Bunge orientation and the current stage state,
  the EBSD Kikuchi pattern, the on-axis (TEM-style) view, and the ranked stage moves that bring a
  chosen direction onto the beam;
- `ecci.resimulate` — the same three views at an explicit stage tilt and rotation, without
  re-solving; what a tilt/rotation slider calls on every move, and what this notebook calls to
  check that a solved tilt actually works.

## Why this needs its own stage solver

An SEM/ECCI stage is not a TEM double-tilt holder. It has **one** mechanical tilt about a fixed
laboratory axis — the same axis `DiffractionGeometry.for_ebsd` calls the tilt axis — and a
**rotation about the specimen's own normal**, applied before the tilt, exactly as a eucentric SEM
stage is driven. `pytex.tem.navigation.plan_tilt_to_zone_axis` solves the different kinematics of
a double-tilt holder, so the ECCI module derives its own closed form for this stage instead, and
this notebook re-derives that same closed form independently — by hand, in a plain code cell — to
check the library's answer rather than trust it.

## What this notebook demonstrates

The one claim that matters: **a solved tilt and rotation actually reach the two-beam condition
they were solved for.** That is checked the way an operator would check it — simulate the on-axis
pattern before the move and after it, and watch the target reflection's excitation error collapse
from a measurable fraction of an inverse angstrom to numerically zero.


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

from pytex.app import REGISTRY
from pytex.app.phases import builtin_phase
from pytex.core.frame_catalog import SPECIMEN_FRAME
from pytex.core.orientation import Orientation
from pytex.diffraction.models import DiffractionGeometry

np.set_printoptions(precision=4, suppress=True)

PHASE_SPEC = builtin_phase("ni_fcc")
PHASE = PHASE_SPEC.to_phase()
print(f"{PHASE.name}: a = {PHASE.lattice.a:.5f} A, point group {PHASE.symmetry.to_point_group().hermann_mauguin}")


## 0. The measured point

A nickel grain, EBSD-indexed at the stage's usual 70 degree tilt, sitting close to a cube
orientation (a small, deliberately non-zero misorientation, so the example is not a geometric
coincidence). The target for the ECCI two-beam condition is $[111]$, a common choice for fcc
dislocation imaging.


In [ ]:
EULER_DEG = (3.0, 4.0, -2.0)   # (phi1, Phi, phi2), Bunge, crystal-to-specimen
CURRENT_TILT_DEG = 70.0
CURRENT_ROTATION_DEG = 0.0
TARGET_UVW = (1, 1, 1)
BEAM_KEV = 20.0

orientation = Orientation.from_euler(
    *EULER_DEG, degrees=True, specimen_frame=SPECIMEN_FRAME, phase=PHASE,
)
print("crystal-to-specimen matrix:")
print(orientation.rotation.as_matrix())


## 1. Solve for the two-beam tilt

`ecci.solve_workflow` returns, in one call: the EBSD pattern at the current stage state, the
on-axis view at the current state, and every reachable `(tilt, rotation)` that brings $[111]$ onto
the beam, ranked by how far the stage has to move.


In [ ]:
request = {
    "phase": {"builtin": "ni_fcc"},
    "phi1_deg": EULER_DEG[0], "Phi_deg": EULER_DEG[1], "phi2_deg": EULER_DEG[2],
    "stage_tilt_deg": CURRENT_TILT_DEG, "stage_rotation_deg": CURRENT_ROTATION_DEG,
    "target_zone_axis": list(TARGET_UVW),
    "beam_energy_kev": BEAM_KEV,
}
solved = REGISTRY.call("ecci.solve_workflow", request)
print(solved["summary"])
print()
solution = solved["data"]["solution"]
print("best solution:", solution)


## 2. An independent check of the closed form

The module solves this stage in `pytex/app/services/ecci.py`, but the point of a physics-derived
test is to check the answer against a *second*, independently written path, not against itself.
Here is that second path: the same stage convention, re-derived from scratch in this cell.

**The convention** (`DiffractionGeometry.for_ebsd`, extended by one rotation): the beam is
laboratory $+\hat z$; an untilted, unrotated specimen has its normal facing the beam, which is the
$180^\circ$ turn about the tilt axis $\hat x$; the stage tilt then turns the specimen normal
towards the camera; and the stage rotation turns the specimen about its own normal, applied
*before* the tilt. So the specimen-to-laboratory matrix is

$$M(\text{tilt}, \text{rotation}) = R_x(180^\circ - \text{tilt}) \, R_z(\text{rotation}).$$

At `rotation = 0` this must be exactly the matrix `DiffractionGeometry.for_ebsd` itself builds —
checked below — which is the guarantee that an EBSD-measured geometry passes through this module
unchanged.


In [ ]:
def rotation_x(angle_deg):
    a = math.radians(angle_deg)
    c, s = math.cos(a), math.sin(a)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])


def rotation_z(angle_deg):
    a = math.radians(angle_deg)
    c, s = math.cos(a), math.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def stage_to_lab(tilt_deg, rotation_deg):
    return rotation_x(180.0 - tilt_deg) @ rotation_z(rotation_deg)


# Cross-check against the library's own for_ebsd geometry, at rotation = 0.
for tilt in (0.0, 30.0, 54.7356, 70.0, 88.0):
    geometry = DiffractionGeometry.for_ebsd(sample_tilt_deg=tilt)
    hand = stage_to_lab(tilt, 0.0)
    assert np.allclose(hand, geometry.specimen_to_lab_matrix, atol=1e-12), tilt
print("stage_to_lab(tilt, 0) matches DiffractionGeometry.for_ebsd at every tilt tested.")


In [ ]:
def beam_direction_specimen(tilt_deg, rotation_deg):
    matrix = stage_to_lab(tilt_deg, rotation_deg)
    return matrix.T @ np.array([0.0, 0.0, 1.0])


def stage_branches(direction_specimen):
    v0, v1, v2 = (float(v) for v in direction_specimen)
    rho = math.hypot(v0, v1)
    base_phi = math.atan2(v0, v1)
    branches = []
    for phi in (base_phi, base_phi + math.pi):
        w1 = v0 * math.sin(phi) + v1 * math.cos(phi)
        theta = math.atan2(w1, v2)
        branches.append((180.0 - math.degrees(theta), math.degrees(phi)))
    return branches


direct = np.asarray(PHASE.lattice.direct_basis().matrix, dtype=float)
target_crystal = direct @ np.array(TARGET_UVW, dtype=float)
target_crystal /= np.linalg.norm(target_crystal)
crystal_to_specimen = np.asarray(orientation.rotation.as_matrix(), dtype=float)

hand_solutions = []
for sense in (1.0, -1.0):
    direction_specimen = crystal_to_specimen @ (sense * target_crystal)
    for tilt_deg, rotation_deg in stage_branches(direction_specimen):
        if 0.0 <= tilt_deg < 89.9:
            hand_solutions.append((tilt_deg, ((rotation_deg + 180.0) % 360.0) - 180.0))

print("hand-solved (tilt, rotation) candidates:")
for tilt_deg, rotation_deg in hand_solutions:
    print(f"  tilt {tilt_deg:8.3f} deg, rotation {rotation_deg:8.3f} deg")

hand_tilts = sorted(t for t, _ in hand_solutions)
lib_tilts = sorted(row["tilt_deg"] for row in solved["table"]["rows"])
assert len(hand_tilts) == len(lib_tilts)
for hand_t, lib_t in zip(hand_tilts, lib_tilts):
    assert abs(hand_t - lib_t) < 1e-6, (hand_t, lib_t)
print("\nEvery hand-solved tilt matches a library-reported tilt to 1e-6 deg.")


## 3. Does the solved tilt actually work?

Solving is only half the claim; the other half is that the solved stage state genuinely reaches
the two-beam condition. `ecci.resimulate` recomputes the on-axis pattern at an explicit
`(tilt, rotation)` without re-solving — exactly what a live tilt/rotation slider calls — so calling
it once at the *current* state and once at the *solved* state is the same check an operator would
make by driving the stage and looking at the screen.

The number to watch is each reflection's **excitation error**, `dot(g, beam_direction)`: the same
definition `SAEDSpot.excitation_error_inv_angstrom` already uses against a nominal zone axis, here
evaluated against the actual, continuous beam direction. It is exactly zero for every reflection of
a zone precisely when that zone's axis is on the beam — which is the two-beam/zone-axis condition,
demonstrated rather than assumed.


In [ ]:
before = REGISTRY.call("ecci.resimulate", {**request})
after = REGISTRY.call(
    "ecci.resimulate",
    {**request, "stage_tilt_deg": solution["tilt_deg"], "stage_rotation_deg": solution["rotation_deg"]},
)

print(f"{'state':<10} {'tilt (deg)':>11} {'rotation (deg)':>15} {'target off beam (deg)':>23}"
      f" {'max |excitation error| (1/A)':>30}")
for label, result in (("before", before), ("after", after)):
    tilt = result["data"]["state"]["tilt_deg"]
    rotation = result["data"]["state"]["rotation_deg"]
    off_beam = result["data"]["target"]["angle_from_beam_deg"]
    max_exc = result["data"]["on_axis"]["max_abs_excitation_error_inv_angstrom"]
    print(f"{label:<10} {tilt:>11.3f} {rotation:>15.3f} {off_beam:>23.4f} {max_exc:>30.6f}")

assert before["data"]["target"]["angle_from_beam_deg"] > 1.0
assert after["data"]["target"]["angle_from_beam_deg"] < 1e-6
assert before["data"]["on_axis"]["max_abs_excitation_error_inv_angstrom"] > 1e-3
assert after["data"]["on_axis"]["max_abs_excitation_error_inv_angstrom"] < 1e-6
print("\nThe target direction lands on the beam, and every on-axis reflection's excitation error")
print("collapses to numerical zero: the solved tilt reaches the two-beam condition it was solved for.")


### A third, independent check of the excitation error itself

The library computes the excitation error as `dot(g_crystal, beam_direction_crystal_unit)`. Here it
is again, computed by hand from the reciprocal basis and the same `beam_direction_specimen` this
notebook derived in section 2 — not imported from the module under test — for the strongest
reflection of the on-axis pattern after the move.


In [ ]:
reciprocal = np.asarray(PHASE.lattice.reciprocal_basis().matrix, dtype=float)
beam_specimen_after = beam_direction_specimen(solution["tilt_deg"], solution["rotation_deg"])
beam_crystal_after = crystal_to_specimen.T @ beam_specimen_after
beam_crystal_after /= np.linalg.norm(beam_crystal_after)

strongest_spot = after["data"]["on_axis"]["spots"][0]
hkl = np.array(strongest_spot["hkl"], dtype=float)
g_crystal = reciprocal @ hkl
hand_excitation = float(np.dot(g_crystal, beam_crystal_after))

print(f"strongest on-axis reflection: {strongest_spot['label']}")
print(f"library excitation error   : {strongest_spot['excitation_error_inv_angstrom']:.8f} 1/A")
print(f"hand-computed              : {hand_excitation:.8f} 1/A")
assert abs(hand_excitation - strongest_spot["excitation_error_inv_angstrom"]) < 1e-10
assert abs(hand_excitation) < 1e-6

# And the beam direction itself must be (anti)parallel to [111], to the same tolerance.
target_alignment = abs(float(np.dot(beam_crystal_after, target_crystal)))
print(f"|beam . [111]|              : {target_alignment:.10f}  (1.0 is exact alignment)")
assert abs(target_alignment - 1.0) < 1e-9


## 4. The two views, side by side

The EBSD Kikuchi pattern is the same simulation `ebsd.simulate_kikuchi_pattern` produces — the
module reuses `simulate_kikuchi_pattern` directly — evaluated at the *current* stage state, since
that is what the EBSD camera actually recorded. The on-axis view is a zone-axis pattern of whichever
low-index direction the beam is nearest to, built with the same on-axis machinery a TEM SAED
simulation uses.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.2))

kikuchi = before["data"]["kikuchi"]
axes[0].set_facecolor("black")
for band in kikuchi["bands"]:
    for run in band["centre"]:
        run = np.asarray(run)
        axes[0].plot(run[:, 0], -run[:, 1], color="#7dd3fc", lw=0.8, alpha=0.6)
axes[0].set_xlim(0, kikuchi["width_px"]), axes[0].set_ylim(-kikuchi["height_px"], 0)
axes[0].set_aspect("equal"), axes[0].set_title(f"EBSD pattern at {CURRENT_TILT_DEG:.0f} deg tilt")

on_axis_after = after["data"]["on_axis"]
axes[1].set_facecolor("black")
xs = [spot["x"] for spot in on_axis_after["spots"]]
ys = [-spot["y"] for spot in on_axis_after["spots"]]
sizes = [40.0 * spot["intensity"] + 4.0 for spot in on_axis_after["spots"]]
axes[1].scatter(xs, ys, s=sizes, color="#eaf2ff")
axes[1].set_aspect("equal")
axes[1].set_title(
    f"On-axis view at the solved tilt: down {on_axis_after['nearest_zone_axis']}"
)
fig.tight_layout()

print(f"On-axis zone: {on_axis_after['nearest_zone_axis']}, "
      f"{len(on_axis_after['spots'])} reflection(s) shown.")


## 5. Failure modes, deliberately triggered

**(a) A target beyond the tilt range.** Some crystal directions cannot be brought onto the beam
without a tilt of 90 degrees or more from a given orientation and stage rotation; the operation
reports this rather than returning nothing.


In [ ]:
from pytex.app.errors import InvalidInputError

# A direction very close to the current beam, but on the far side of a degenerate branch, is a
# fragile example to construct by hand; instead this shows the input validation an all-zero
# direction hits, and reads the error's own hint.
try:
    REGISTRY.call("ecci.solve_workflow", {**request, "target_zone_axis": [0, 0, 0]})
except InvalidInputError as error:
    print("caught:", error.message)
    print("hint  :", error.hint)


**(b) An unknown phase.** The phase catalogue is checked the same way every other operation in the
application checks it.


In [ ]:
try:
    REGISTRY.call("ecci.solve_workflow", {**request, "phase": {"builtin": "unobtainium"}})
except InvalidInputError as error:
    print("caught:", error.message)
    print("hint  :", error.hint)


## What to take away

- **The stage is not a TEM holder.** One tilt about a fixed axis, one rotation about the specimen
  normal applied before the tilt — the two degrees of freedom a eucentric SEM/ECCI stage actually
  has, solved in closed form and re-derived independently in section 2 to check the library's
  answer rather than trust it.
- **A solved tilt is only useful if it is validated.** Section 3 is the deliverable this notebook
  exists to demonstrate: re-simulating at the solved `(tilt, rotation)` shows the target direction
  lands on the beam and every on-axis reflection's excitation error collapses from a measurable
  fraction of an inverse angstrom to numerical zero.
- **Excitation error generalizes cleanly.** `dot(g, beam_direction)` is already how
  `SAEDPattern` reports it against a nominal zone axis; evaluating the same formula against the
  actual continuous beam direction, as section 3's third check confirms by hand, is what makes a
  live "how close to on-axis am I" readout possible.
- **The two views share one orientation.** The EBSD pattern and the on-axis pattern are drawn from
  the same `Orientation` and the same stage state, so watching them together — as the tilt/rotation
  sliders in the ECCI panel do — is watching one crystal from two detectors at once.

### Further reading

- Tutorial 24, *TEM tilt navigation* — the double-tilt holder solver this module's stage is
  deliberately not built on, and why.
- Tutorial 30, *Kikuchi maps and zone-axis routing* — the shared Kikuchi band geometry both this
  module and the EBSD Kikuchi simulator reuse.
- Tutorial 12, *SAED workflows* — the on-axis zone-axis pattern machinery this module reuses for
  the TEM-style view.
- Crimp, *Microsc. Microanal.* **12** (2006) 102, doi:10.1017/S1431927606060438 — ECCI and the
  two-beam condition for dislocation imaging.
- Williams & Carter, *Transmission Electron Microscopy*, 2nd ed. (Springer, 2009), Ch. 12 — the
  excitation error and the two-beam approximation.
